# Eval 代码裁切流程分析
演示 eval_hrbench.py 中 bbox 裁切的完整过程，展示坐标偏移问题

In [1]:
import os, json, re, math, base64, io
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

DATA_DIR = '/mdr5/user/quantaalpha/hutu/DeepEyes/data/HR-Bench'
RESULT_DIR = '/mdr5/user/quantaalpha/hutu/DeepEyes/results/HR-Bench/deepeyes'
TEST_TYPE = 'hr_bench_4k'  # 'hr_bench_4k' or 'hr_bench_8k'
MODEL_NAME = 'deepeyes'

# 加载数据
df = pd.read_csv(os.path.join(DATA_DIR, f'{TEST_TYPE}.tsv'), sep='\t')
with open(os.path.join(RESULT_DIR, f'result_{TEST_TYPE}_{MODEL_NAME}.jsonl')) as f:
    results = [json.loads(line) for line in f]
q2row = {row['question']: i for i, row in df.iterrows()}
print(f'Loaded {len(results)} results, {len(df)} images')

Loaded 800 results, 800 images


In [2]:
# ===== eval 代码中的 smart_resize（原样搬来）=====
IMAGE_FACTOR = 28
MIN_PIXELS = 4 * 28 * 28
MAX_PIXELS = 16384 * 28 * 28

def round_by_factor(number, factor):
    return round(number / factor) * factor

def ceil_by_factor(number, factor):
    return math.ceil(number / factor) * factor

def floor_by_factor(number, factor):
    return math.floor(number / factor) * factor

def smart_resize(height, width, factor=IMAGE_FACTOR, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS):
    """注意：函数签名是 (height, width)"""
    h_bar = max(factor, round_by_factor(height, factor))
    w_bar = max(factor, round_by_factor(width, factor))
    if h_bar * w_bar > max_pixels:
        beta = math.sqrt((height * width) / max_pixels)
        h_bar = floor_by_factor(height / beta, factor)
        w_bar = floor_by_factor(width / beta, factor)
    elif h_bar * w_bar < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h_bar = ceil_by_factor(height * beta, factor)
        w_bar = ceil_by_factor(width * beta, factor)
    return h_bar, w_bar

def decode_image(b64):
    return Image.open(io.BytesIO(base64.b64decode(b64))).convert('RGB')

def extract_tool_calls(text):
    bboxes = []
    for tc in re.findall(r'<tool_call>\s*(.*?)\s*(?:</tool_call>|$)', text, re.DOTALL):
        try:
            call = json.loads(tc.strip())
            if 'arguments' in call and 'bbox_2d' in call['arguments']:
                bboxes.append(call['arguments']['bbox_2d'])
        except: pass
    return bboxes

In [3]:
def show_crop_pipeline(idx):
    """
    完整演示 eval_hrbench.py 对第 idx 个样本的裁切过程：
    
    Step 1: 原图 -> smart_resize -> 发给模型
    Step 2: 模型输出 bbox（在 resized 坐标空间）
    Step 3: eval 代码直接在原图上 crop（坐标未映射）
    Step 4: crop 结果再 smart_resize
    
    对比：如果做了正确的坐标映射会怎样
    """
    r = results[idx]
    row_idx = q2row.get(r['question'])
    if row_idx is None:
        print('Cannot find image'); return
    
    ori_img = decode_image(df.iloc[row_idx]['image'])
    ori_w, ori_h = ori_img.size
    
    # 收集所有 bbox
    all_bboxes = []
    for msg in r['pred_output']:
        if msg['role'] == 'assistant':
            content = msg['content'] if isinstance(msg['content'], str) else ''
            all_bboxes.extend(extract_tool_calls(content))
    
    if not all_bboxes:
        print(f'Sample {idx}: no tool call'); return
    
    # ========== Step 1: eval 代码对原图的处理 ==========
    # eval 代码: resize_w, resize_h = smart_resize(ori_width, ori_height)
    # 注意: 把 width 当 height 传了！
    resize_w, resize_h = smart_resize(ori_w, ori_h)  # 复现 eval 的 bug
    resized_img = ori_img.resize((resize_w, resize_h), resample=Image.BICUBIC)
    
    scale_w = ori_w / resize_w
    scale_h = ori_h / resize_h
    
    print(f'===== Sample {idx} =====')
    print(f'Question: {r["question"]}')
    print(f'GT: {r["answer"]}  Pred: {r["pred_ans"]}')
    print(f'Original: {ori_w}x{ori_h}')
    print(f'Model sees: {resize_w}x{resize_h} (scale_w={scale_w:.4f}, scale_h={scale_h:.4f})')
    print()
    
    for ti, bbox in enumerate(all_bboxes):
        left, top, right, bottom = bbox
        crop_w, crop_h = right - left, bottom - top
        
        # 正确映射到原图的坐标
        mapped = [int(left * scale_w), int(top * scale_h),
                  int(right * scale_w), int(bottom * scale_h)]
        
        print(f'--- Tool Call {ti+1} ---')
        print(f'Model output bbox: {bbox}  (in {resize_w}x{resize_h} space)')
        print(f'Mapped to original: {mapped}  (in {ori_w}x{ori_h} space)')
        print(f'Offset: dx={mapped[0]-left}, dy={mapped[1]-top}')
        print()
        
        # eval 的 crop（直接用 model bbox 裁原图）
        eval_crop = ori_img.crop((left, top, right, bottom))
        # eval 的 smart_resize on crop（又把宽当高传了）
        eval_new_w, eval_new_h = smart_resize(crop_w, crop_h)
        eval_crop_resized = eval_crop.resize((eval_new_w, eval_new_h), resample=Image.BICUBIC)
        
        # 正确的 crop（映射后的坐标）
        correct_crop = ori_img.crop(tuple(mapped))
        
        # 从 resized 图上裁（模型真正想看的）
        model_view_crop = resized_img.crop((left, top, right, bottom))
        
        # ========== 可视化 ==========
        fig = plt.figure(figsize=(20, 10))
        
        # Row 1: 原图 + 两个 bbox 对比
        ax1 = fig.add_subplot(2, 4, (1, 2))
        ax1.imshow(ori_img)
        # eval 的 bbox（红色）- 直接用 model 坐标
        rect1 = patches.Rectangle((left, top), crop_w, crop_h,
                                   linewidth=2, edgecolor='red', facecolor='none', linestyle='-')
        ax1.add_patch(rect1)
        ax1.text(left, top-10, f'eval bbox (wrong)', color='red', fontsize=10,
                 bbox=dict(facecolor='white', alpha=0.8))
        # 正确映射的 bbox（绿色）
        rect2 = patches.Rectangle((mapped[0], mapped[1]), mapped[2]-mapped[0], mapped[3]-mapped[1],
                                   linewidth=2, edgecolor='lime', facecolor='none', linestyle='--')
        ax1.add_patch(rect2)
        ax1.text(mapped[0], mapped[3]+20, f'correct bbox (mapped)', color='lime', fontsize=10,
                 bbox=dict(facecolor='black', alpha=0.8))
        ax1.set_title(f'Original {ori_w}x{ori_h}\nRed=eval(wrong) Green=correct(mapped)', fontsize=11)
        ax1.axis('off')
        
        # Row 1: resized 图 + model bbox
        ax2 = fig.add_subplot(2, 4, (3, 4))
        ax2.imshow(resized_img)
        rect3 = patches.Rectangle((left, top), crop_w, crop_h,
                                   linewidth=2, edgecolor='cyan', facecolor='none')
        ax2.add_patch(rect3)
        ax2.set_title(f'Model sees {resize_w}x{resize_h}\nBbox {bbox}', fontsize=11)
        ax2.axis('off')
        
        # Row 2: 三种 crop 对比
        ax3 = fig.add_subplot(2, 4, 5)
        ax3.imshow(eval_crop)
        ax3.set_title(f'Eval crop (wrong pos)\nori_img.crop({bbox})\n{eval_crop.size}', fontsize=9, color='red')
        ax3.axis('off')
        
        ax4 = fig.add_subplot(2, 4, 6)
        ax4.imshow(eval_crop_resized)
        ax4.set_title(f'Eval crop + resize\nsmart_resize({crop_w},{crop_h})\n-> {eval_new_w}x{eval_new_h}', fontsize=9, color='red')
        ax4.axis('off')
        
        ax5 = fig.add_subplot(2, 4, 7)
        ax5.imshow(correct_crop)
        ax5.set_title(f'Correct crop (mapped)\nori_img.crop({mapped})\n{correct_crop.size}', fontsize=9, color='green')
        ax5.axis('off')
        
        ax6 = fig.add_subplot(2, 4, 8)
        ax6.imshow(model_view_crop)
        ax6.set_title(f'Model intended\nresized_img.crop({bbox})\n{model_view_crop.size}', fontsize=9, color='blue')
        ax6.axis('off')
        
        plt.suptitle(f'Sample {idx} - Tool Call {ti+1} | Offset: dx={mapped[0]-left} dy={mapped[1]-top}', fontsize=13)
        plt.tight_layout()
        plt.show()
        print()

In [ ]:
show_crop_pipeline(42)

===== Sample 42 =====
Question: What is the text written above the logo on the jacket's sleeve?
GT: C  Pred: C. epio MIX
Original: 4032x4032
Model sees: 3584x3584 (scale_w=1.1250, scale_h=1.1250)

--- Tool Call 1 ---
Model output bbox: [1732, 1650, 1898, 1830]  (in 3584x3584 space)
Mapped to original: [1948, 1856, 2135, 2058]  (in 4032x4032 space)
Offset: dx=216, dy=206



In [5]:
# 批量统计偏移量
offsets = []
for idx, r in enumerate(results):
    row_idx = q2row.get(r['question'])
    if row_idx is None: continue
    b64 = df.iloc[row_idx]['image']
    img = Image.open(io.BytesIO(base64.b64decode(b64)))
    ori_w, ori_h = img.size
    resize_w, resize_h = smart_resize(ori_w, ori_h)
    scale_w, scale_h = ori_w / resize_w, ori_h / resize_h
    
    for msg in r['pred_output']:
        if msg['role'] != 'assistant': continue
        content = msg['content'] if isinstance(msg['content'], str) else ''
        for bbox in extract_tool_calls(content):
            left, top, right, bottom = bbox
            dx = abs(left * scale_w - left)
            dy = abs(top * scale_h - top)
            offsets.append({
                'idx': idx, 'ori': f'{ori_w}x{ori_h}',
                'resize': f'{resize_w}x{resize_h}',
                'scale_w': scale_w, 'scale_h': scale_h,
                'dx': dx, 'dy': dy,
                'bbox_w': right-left, 'bbox_h': bottom-top,
            })

print(f'Total tool calls: {len(offsets)}')
if offsets:
    dxs = [o['dx'] for o in offsets]
    dys = [o['dy'] for o in offsets]
    print(f'dx offset: min={min(dxs):.0f} max={max(dxs):.0f} mean={sum(dxs)/len(dxs):.0f} px')
    print(f'dy offset: min={min(dys):.0f} max={max(dys):.0f} mean={sum(dys)/len(dys):.0f} px')
    print()
    # 按 scale 分布
    from collections import Counter
    scale_dist = Counter(f"{o['scale_w']:.3f}x{o['scale_h']:.3f}" for o in offsets)
    print('Scale distribution:')
    for s, cnt in scale_dist.most_common(10):
        print(f'  {s}: {cnt} tool calls')

Total tool calls: 1452
dx offset: min=0 max=432 mean=148 px
dy offset: min=0 max=419 mean=169 px

Scale distribution:
  1.125x1.125: 1009 tool calls
  1.000x1.005: 83 tool calls
  1.000x0.995: 68 tool calls
  1.000x1.000: 66 tool calls
  1.000x1.003: 42 tool calls
  1.000x1.002: 24 tool calls
  1.000x0.993: 21 tool calls
  1.000x0.999: 21 tool calls
  1.000x1.004: 17 tool calls
  1.000x0.997: 16 tool calls


In [ ]:
# 多看几个样本
# show_crop_pipeline(0)
# show_crop_pipeline(42)